# 모델 평가 시각화

**전제**: `05_train_lstm_improved2.ipynb` 실행 완료 후 `data/predictions.npz` 가 업데이트된 상태

| 시각화 | 설명 |
|--------|------|
| Performance Bar Chart | Accuracy / Precision / Recall / F1 비교 (랜덤 베이스라인 포함) |
| Per-Class F1 | Down / Up 클래스별 F1 분리 비교 |
| Model Agreement | 모델 합의 수에 따른 Up Precision vs Coverage |
| Confusion Matrix | 모델별 혼동 행렬 |
| Prediction Distribution | 모델별 상승/하락 예측 비율 |


In [ ]:
# [STEP 1] Google Drive 마운트 & 경로 설정
import os

DRIVE_PROJECT_PATH = "/content/drive/MyDrive/term_project"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = DRIVE_PROJECT_PATH
    print(f"[Drive 마운트 완료] BASE_DIR={BASE_DIR}")
except Exception:
    BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
    print(f"[로컬 환경] BASE_DIR={BASE_DIR}")

DATA_DIR   = os.path.join(BASE_DIR, "data")
RESULT_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULT_DIR, exist_ok=True)

pred_path = os.path.join(DATA_DIR, "predictions.npz")
print(f"predictions.npz 존재: {os.path.exists(pred_path)}")

In [ ]:
# [STEP 2] 라이브러리 & matplotlib 한글 설정
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

# Colab 한글 폰트 설정
try:
    import subprocess
    subprocess.run(["apt-get", "install", "-y", "-q", "fonts-nanum"], check=True,
                   capture_output=True)
    fm._load_fontmanager(try_read_cache=False)
    plt.rcParams['font.family'] = 'NanumGothic'
    print("한글 폰트 설정 완료 (NanumGothic)")
except Exception:
    print("한글 폰트 설치 실패 → 기본 폰트 사용")

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

In [ ]:
# [STEP 3] predictions.npz 로드
assert os.path.exists(pred_path), (
    f"predictions.npz 없음: {pred_path}\n"
    "05_train_lstm_improved2.ipynb 를 먼저 완료해주세요."
)

pred_data = np.load(pred_path)
y_test    = pred_data["y_test"]

MODEL_NAMES = [n for n in ["arima", "lstm", "transformer"] if n in pred_data.files]

print(f"로드 완료: {pred_path}")
print(f"총 샘플 수: {len(y_test):,}")
print(f"실제 Up 비율: {y_test.mean():.3f}")
print(f"포함된 모델: {MODEL_NAMES}")
print()
for name in MODEL_NAMES:
    preds = pred_data[name]
    print(f"  {name.upper():12s} Up 예측 비율: {preds.mean():.3f}  "
          f"(Down={( preds==0).sum():,} / Up={(preds==1).sum():,})")

In [ ]:
# [STEP 4] 모델별 성능 지표 계산
rows = []
for name in MODEL_NAMES:
    y_pred = pred_data[name]
    f1_per_class = f1_score(y_test, y_pred, average=None, zero_division=0)
    rows.append({
        "Model":      name.upper(),
        "Accuracy":   accuracy_score(y_test, y_pred),
        "Precision":  precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall":     recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1 (weighted)": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1 (macro)":    f1_score(y_test, y_pred, average="macro",    zero_division=0),
        "F1 Down":    f1_per_class[0] if len(f1_per_class) > 0 else 0.0,
        "F1 Up":      f1_per_class[1] if len(f1_per_class) > 1 else 0.0,
        "Up Ratio":   float(y_pred.mean()),
    })

df_metrics = pd.DataFrame(rows)
display(df_metrics.style
    .format({c: "{:.4f}" for c in df_metrics.columns if c != "Model"})
    .highlight_max(subset=["Accuracy", "F1 (macro)", "F1 Up"], color="#d4edda")
    .set_caption("Model Performance Summary")
)

In [ ]:
# [PLOT 1] Accuracy / Precision / Recall / F1 비교 막대그래프
metric_cols = ["Accuracy", "Precision", "Recall", "F1 (weighted)"]
x     = np.arange(len(df_metrics))
width = 0.18
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

fig, ax = plt.subplots(figsize=(10, 5))
for i, (col, color) in enumerate(zip(metric_cols, colors)):
    ax.bar(x + (i - 1.5) * width, df_metrics[col], width, label=col, color=color)

ax.axhline(0.5, color="red", linestyle="--", linewidth=1.3, label="Random Baseline")
ax.set_xticks(x)
ax.set_xticklabels(df_metrics["Model"], fontsize=12)
ax.set_ylim(0.4, 0.75)
ax.set_ylabel("Score")
ax.set_title("Model Classification Performance")
ax.legend()
plt.tight_layout()

save_path = os.path.join(RESULT_DIR, "01_performance_bar.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {save_path}")

In [ ]:
# [PLOT 2] Down / Up 클래스별 F1 분리 비교
x     = np.arange(len(df_metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars_down = ax.bar(x - width / 2, df_metrics["F1 Down"], width, label="Down", color="#4C72B0")
bars_up   = ax.bar(x + width / 2, df_metrics["F1 Up"],   width, label="Up",   color="#DD8452")

for bar in list(bars_down) + list(bars_up):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.006,
        f"{bar.get_height():.3f}",
        ha="center", va="bottom", fontsize=9,
    )

ax.axhline(0.5, color="red", linestyle="--", linewidth=1.3, label="Random Baseline")
ax.set_xticks(x)
ax.set_xticklabels(df_metrics["Model"], fontsize=12)
ax.set_ylim(0, 0.85)
ax.set_ylabel("F1-score")
ax.set_title("Per-Class F1-score: Down vs Up")
ax.legend()
plt.tight_layout()

save_path = os.path.join(RESULT_DIR, "02_per_class_f1.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {save_path}")

In [ ]:
# [PLOT 3] 모델 합의 분석 — Up Precision vs Coverage
agree_counts = sum(pred_data[n] for n in MODEL_NAMES)

labels    = [f"≥1 Up ({len(MODEL_NAMES)}개 중 1)",
             f"≥2 Up ({len(MODEL_NAMES)}개 중 2)",
             f"=3 Up (전체 동의)"]
thresholds = [1, 2, 3]

precisions, coverages, sample_counts = [], [], []
for t in thresholds:
    mask = agree_counts >= t
    n    = int(mask.sum())
    sample_counts.append(n)
    precisions.append(float(y_test[mask].mean()) if n > 0 else 0.0)
    coverages.append(float(mask.mean()))

x     = np.arange(len(labels))
width = 0.35

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

bars = ax1.bar(x - width / 2, precisions, width, label="Up Precision", color="#55A868")
ax2.bar(x + width / 2, coverages, width, label="Coverage", color="#C44E52", alpha=0.7)

for bar, val, n in zip(bars, precisions, sample_counts):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f"{val:.3f}\n(n={n:,})", ha="center", va="bottom", fontsize=8)

ax1.axhline(0.5, color="red", linestyle="--", linewidth=1.3, label="Random Baseline")
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=10)
ax1.set_ylim(0, 0.85)
ax1.set_ylabel("Up Precision")
ax2.set_ylim(0, 1)
ax2.set_ylabel("Coverage (sample ratio)")
ax1.set_title("Model Agreement: Up Precision vs Coverage")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper right")
plt.tight_layout()

save_path = os.path.join(RESULT_DIR, "03_model_agreement.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {save_path}")

In [ ]:
# [PLOT 4] 모델별 Confusion Matrix
n_models = len(MODEL_NAMES)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, name in zip(axes, MODEL_NAMES):
    cm   = confusion_matrix(y_test, pred_data[name])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Down", "Up"])
    disp.plot(values_format=",d", ax=ax, colorbar=False)
    ax.set_title(f"{name.upper()} Confusion Matrix", fontsize=12)

plt.tight_layout()
save_path = os.path.join(RESULT_DIR, "04_confusion_matrices.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {save_path}")

In [ ]:
# [PLOT 5] 모델별 상승/하락 예측 비율 (실제값 기준선 포함)
actual_up_ratio = float(y_test.mean())

model_labels = [n.upper() for n in MODEL_NAMES]
down_ratios  = [(pred_data[n] == 0).mean() for n in MODEL_NAMES]
up_ratios    = [(pred_data[n] == 1).mean() for n in MODEL_NAMES]

x     = np.arange(len(model_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width / 2, down_ratios, width, label="Down", color="#4C72B0")
ax.bar(x + width / 2, up_ratios,   width, label="Up",   color="#DD8452")

ax.axhline(actual_up_ratio, color="#DD8452", linestyle="--", linewidth=1.5,
           label=f"Actual Up ratio ({actual_up_ratio:.3f})")
ax.axhline(1 - actual_up_ratio, color="#4C72B0", linestyle="--", linewidth=1.5,
           label=f"Actual Down ratio ({1-actual_up_ratio:.3f})")

ax.set_xticks(x)
ax.set_xticklabels(model_labels, fontsize=12)
ax.set_ylim(0, 1)
ax.set_ylabel("Prediction Ratio")
ax.set_title("Prediction Distribution by Model")
ax.legend()
plt.tight_layout()

save_path = os.path.join(RESULT_DIR, "05_prediction_distribution.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"저장: {save_path}")

In [ ]:
# [STEP 5] 결과 CSV 저장
csv_path = os.path.join(RESULT_DIR, "model_metrics.csv")
df_metrics.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"성능 요약 저장: {csv_path}")
print()
print("=== 저장된 시각화 파일 ===")
for f in sorted(os.listdir(RESULT_DIR)):
    if f.endswith(".png"):
        print(f"  {f}")